# Ask the Energiewende agent

Type a question about electricity prices, power generation, weather or Bundestag energy policy and press **Ask**.
The agent picks the right data source by itself and lists its sources.

Run all cells once (*Run → Run All*). In VS Code, pick the `.venv` Python as kernel (top right). Needs the search index (`scripts/build_index.py`) and the LLM settings in `.env`.

In [ ]:
# Load the agent code. The search index loads on the first Bundestag question.
import os
import sys
from pathlib import Path

# Work from the project folder, so ".env", "data/" and "src/" are found.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

from energiewende.agent import graph

In [ ]:
# The question box.
import html

import ipywidgets as widgets
from IPython.display import display

question_box = widgets.Textarea(
    placeholder="z.B. Wie viel Windstrom wurde gestern erzeugt?",
    layout=widgets.Layout(width="80%", height="60px"),
)
ask_button = widgets.Button(description="Ask", button_style="primary")
answer_box = widgets.HTML()  # the answer is shown here


def on_ask(_):
    question = question_box.value.strip()
    if not question:
        return
    answer_box.value = "<i>Thinking ...</i>"
    try:
        result = graph.ask(question)
    except Exception as error:  # show the error instead of hiding it
        answer_box.value = f"<pre style='color: #c0392b;'>Error: {html.escape(repr(error))}</pre>"
        return
    sources = "".join(f"<li>{html.escape(s)}</li>" for s in result["sources"])
    tools_used = ", ".join(call["name"] for call in result["tool_calls"]) or "none"
    answer_box.value = f"""
    <div style="border-left: 4px solid #2a7ae2; padding: 4px 12px;">
      <p style="white-space: pre-wrap;">{html.escape(result["answer"])}</p>
      <small>Tools: {tools_used} · {result["latency_ms"]} ms · {result["tokens"]} tokens</small>
      <ul><small>{sources}</small></ul>
    </div>"""


ask_button.on_click(on_ask)
display(widgets.HBox([question_box, ask_button]), answer_box)